# Healthcare Data Analysis — Medallion Architecture

End-to-end ETL pipeline reading `healthcare_dataset.csv` from ADLS Gen2 and
processing it through **Bronze → Silver → Gold** layers in the `azuredatabricks1` catalog.

| Layer | Schema | Purpose |
|-------|--------|---------|
| **Bronze** | `azuredatabricks1.bronze` | Raw ingestion — data as-is from source |
| **Silver** | `azuredatabricks1.silver` | Cleaned, standardized, enriched |
| **Gold** | `azuredatabricks1.gold` | Aggregated, business-ready analytics |

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks")

# Remove cached module if re-running
if "ADLS_Databricks_Connection" in sys.modules:
    del sys.modules["ADLS_Databricks_Connection"]

import ADLS_Databricks_Connection as adls_module

# Inject notebook-scoped globals into the imported module
adls_module.dbutils = dbutils
adls_module.spark = spark
adls_module.display = display

from ADLS_Databricks_Connection import ADLSManager

print("✔ Imported ADLSManager from ADLS_Databricks_Connection.py")

In [0]:
CONFIG_PATH = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"
adls = ADLSManager(CONFIG_PATH)
adls.configure_oauth()
print("✔ ADLS connection ready")

In [0]:
# Use existing catalog
spark.sql("USE CATALOG azuredatabricks1")
print("✔ Using catalog: azuredatabricks1")

# Create medallion schemas
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"✔ Schema created: azuredatabricks1.{schema}")

print("\n✔ Medallion architecture schemas ready")

## 🟥 Bronze Layer — Raw Ingestion
Ingest raw CSV from ADLS Gen2 into `azuredatabricks1.bronze.healthcare_raw` with no transformations.

In [0]:
# Read raw CSV from ADLS Gen2
df_bronze_raw = adls.read_file("ainput1/healthcare_dataset.csv", "csv", header="true", inferSchema="true")

# Rename columns to snake_case at ingestion (Delta requires no special chars)
column_mapping = {
    "Name": "name",
    "Age": "age",
    "Gender": "gender",
    "Blood Type": "blood_type",
    "Medical Condition": "medical_condition",
    "Date of Admission": "admission_date",
    "Doctor": "doctor",
    "Hospital": "hospital",
    "Insurance Provider": "insurance_provider",
    "Billing Amount": "billing_amount",
    "Room Number": "room_number",
    "Admission Type": "admission_type",
    "Discharge Date": "discharge_date",
    "Medication": "medication",
    "Test Results": "test_results",
}

df_bronze = df_bronze_raw
for old_name, new_name in column_mapping.items():
    df_bronze = df_bronze.withColumnRenamed(old_name, new_name)

print(f"✔ Bronze ingested  |  Rows: {df_bronze.count():,}  |  Columns: {len(df_bronze.columns)}")
df_bronze.printSchema()

In [0]:
# Write raw data to bronze Delta table
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("azuredatabricks1.bronze.healthcare_raw")
)

print("✔ Written to: azuredatabricks1.bronze.healthcare_raw")

In [0]:
df_bronze_verify = spark.table("azuredatabricks1.bronze.healthcare_raw")
print(f"Bronze table row count: {df_bronze_verify.count():,}")
display(df_bronze_verify.limit(10))

## 🟨 Silver Layer — Cleansed and Enriched
Transformations applied:
- Standardize `name` to proper case
- Filter out invalid billing amounts (negative values)
- Remove duplicate records (window-based deduplication)
- Add `length_of_stay` computed column (days between admission and discharge)
- Add `ingestion_timestamp` audit column

In [0]:
from pyspark.sql.functions import (
    col, initcap, datediff, trim, current_timestamp, row_number
)
from pyspark.sql.window import Window

# Read from bronze table
df_raw = spark.table("azuredatabricks1.bronze.healthcare_raw")

# --- Step 1: Standardize name to proper case and trim whitespace ---
df_silver = df_raw.withColumn("name", initcap(trim(col("name"))))

# --- Step 2: Filter out invalid billing amounts ---
before_count = df_silver.count()
df_silver = df_silver.filter(col("billing_amount") >= 0)
after_count = df_silver.count()
print(f"✔ Removed {before_count - after_count:,} rows with negative billing")

# --- Step 3: Remove duplicates ---
window_dedup = Window.partitionBy(
    "name", "age", "gender", "admission_date", "medical_condition"
).orderBy(col("billing_amount").desc())

df_silver = (
    df_silver
    .withColumn("_row_num", row_number().over(window_dedup))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)
print(f"✔ After deduplication: {df_silver.count():,} rows")

# --- Step 4: Add computed columns ---
df_silver = (
    df_silver
    .withColumn("length_of_stay", datediff(col("discharge_date"), col("admission_date")))
    .withColumn("ingestion_timestamp", current_timestamp())
)

print(f"✔ Silver transformation complete  |  Rows: {df_silver.count():,}  |  Columns: {len(df_silver.columns)}")
df_silver.printSchema()

In [0]:
# Write cleaned data to silver Delta table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("azuredatabricks1.silver.healthcare_cleaned")
)

print("✔ Written to: azuredatabricks1.silver.healthcare_cleaned")

In [0]:
df_silver_verify = spark.table("azuredatabricks1.silver.healthcare_cleaned")
print(f"Silver table row count: {df_silver_verify.count():,}")
print(f"\n--- Sample Silver Records ---")
display(df_silver_verify.limit(10))

# Quick data quality check
print("\n--- Null Counts per Column ---")
from pyspark.sql.functions import sum as _sum, when, isnull
null_counts = df_silver_verify.select(
    [_sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in df_silver_verify.columns]
)
display(null_counts)

## 🟩 Gold Layer — Business-Ready Aggregations
Three analytics tables built from the silver layer:
- **Condition Summary** — patient counts, avg billing, avg length of stay by medical condition
- **Admission Summary** — metrics grouped by admission type and insurance provider
- **Hospital Performance** — hospital-level KPIs

In [0]:
from pyspark.sql.functions import (
    count, avg, min as _min, max as _max, round as _round, sum as _sum
)

df_silver_src = spark.table("azuredatabricks1.silver.healthcare_cleaned")

# Aggregate by medical condition
df_gold_condition = (
    df_silver_src
    .groupBy("medical_condition")
    .agg(
        count("*").alias("patient_count"),
        _round(avg("age"), 1).alias("avg_age"),
        _round(avg("billing_amount"), 2).alias("avg_billing"),
        _round(_min("billing_amount"), 2).alias("min_billing"),
        _round(_max("billing_amount"), 2).alias("max_billing"),
        _round(avg("length_of_stay"), 1).alias("avg_length_of_stay"),
        _round(_sum("billing_amount"), 2).alias("total_billing")
    )
    .orderBy(col("patient_count").desc())
)

# Write to gold
(
    df_gold_condition.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("azuredatabricks1.gold.healthcare_condition_summary")
)

print("✔ Written to: azuredatabricks1.gold.healthcare_condition_summary")
display(df_gold_condition)

In [0]:
# Aggregate by admission type and insurance provider
df_gold_admission = (
    df_silver_src
    .groupBy("admission_type", "insurance_provider")
    .agg(
        count("*").alias("patient_count"),
        _round(avg("billing_amount"), 2).alias("avg_billing"),
        _round(avg("length_of_stay"), 1).alias("avg_length_of_stay"),
        _round(_sum("billing_amount"), 2).alias("total_billing")
    )
    .orderBy("admission_type", col("patient_count").desc())
)

# Write to gold
(
    df_gold_admission.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("azuredatabricks1.gold.healthcare_admission_summary")
)

print("✔ Written to: azuredatabricks1.gold.healthcare_admission_summary")
display(df_gold_admission)

In [0]:
from pyspark.sql.functions import countDistinct

# Hospital-level KPIs
df_gold_hospital = (
    df_silver_src
    .groupBy("hospital")
    .agg(
        count("*").alias("total_patients"),
        countDistinct("doctor").alias("unique_doctors"),
        countDistinct("medical_condition").alias("conditions_treated"),
        _round(avg("billing_amount"), 2).alias("avg_billing"),
        _round(avg("length_of_stay"), 1).alias("avg_length_of_stay"),
        _round(_sum("billing_amount"), 2).alias("total_revenue")
    )
    .orderBy(col("total_patients").desc())
)

# Write to gold
(
    df_gold_hospital.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("azuredatabricks1.gold.healthcare_hospital_performance")
)

print("✔ Written to: azuredatabricks1.gold.healthcare_hospital_performance")
display(df_gold_hospital)

In [0]:
print("=" * 60)
print("     Medallion Architecture — Pipeline Summary")
print("=" * 60)

layers = {
    "Bronze — healthcare_raw": "azuredatabricks1.bronze.healthcare_raw",
    "Silver — healthcare_cleaned": "azuredatabricks1.silver.healthcare_cleaned",
    "Gold   — condition_summary": "azuredatabricks1.gold.healthcare_condition_summary",
    "Gold   — admission_summary": "azuredatabricks1.gold.healthcare_admission_summary",
    "Gold   — hospital_performance": "azuredatabricks1.gold.healthcare_hospital_performance",
}

for label, table in layers.items():
    row_count = spark.table(table).count()
    print(f"  {label:40s} | {row_count:>8,} rows")

print("=" * 60)
print("✅ Medallion pipeline completed successfully!")